In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import itertools
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from dateutil.easter import easter
from pmdarima import auto_arima

np.random.seed(42)

# 1.Data Loading - Total Electricity Consumption
energy_consumption = pd.read_csv('Energy_Consumption.csv', sep = ';')
energy_consumption['Consumption'] = energy_consumption['Consumption'].astype(str).str.replace(',', '.').astype(float)
energy_consumption['Period'] = pd.to_datetime(energy_consumption['Period'], dayfirst = True)
energy_consumption.set_index('Period', inplace = True)
energy_consumption.index.freq = 'MS'
print(energy_consumption.tail())

# Chart - Original Data

plt.figure(figsize = (15, 6))

# Plotting the Original series
plt.plot(energy_consumption.index, energy_consumption['Consumption'], color = 'blue', lw = 2, label = 'Energy Consumption - Total (GWh)')
plt.title('Total Energy Consumption in Brazil - Historical Series', fontsize = 14, fontweight = 'bold')
plt.ylabel('Consumption (GWh)')
plt.xlabel('Years')
plt.grid(True, alpha = .2)
plt.legend(loc = 'best', handlelength = 1.5, frameon = False)

plt.text(0.99, -0.12, 'Source: Energy Research Office (EPE)',
         transform = plt.gca().transAxes,
         fontsize = 10,
         color = 'gray',
         style = 'italic',
         horizontalalignment = 'right')

plt.tight_layout()
plt.savefig('energy_consumption_epe.png', dpi = 300, bbox_inches = 'tight')
plt.show()

# 2. Taking the logarithm
log_energy_consumption = np.log(energy_consumption['Consumption'])
print(log_energy_consumption.tail())

# 3. Decomposing the series, using STL model
stl = STL(log_energy_consumption, period = 12, seasonal = 13, robust = True)
stl_log_energy_consumption = stl.fit()

# Plotting the decomposing: Trend + Seasonal + Residuals
fig = stl_log_energy_consumption.plot()
plt.savefig('energy_consumption_decomposition.png', dpi = 300, bbox_inches = 'tight')
plt.show()

# 4. Clearly, the trend is non-stationarity. To endorse it, the ADFULLER function was applied
log_energy_consumption_adf = adfuller(log_energy_consumption)
print(f"ADF p-value: {log_energy_consumption_adf[1]: .4f}") 

# To achieve stationarity, we apply differencing and run the ADF test again
dlog_energy_consumption = (log_energy_consumption).diff().dropna()*100
dlog_energy_consumption_adf = adfuller(dlog_energy_consumption)
print(f"New ADF p-value: {dlog_energy_consumption_adf[1]: .4f}") 
print(dlog_energy_consumption.tail())
dlog_energy_consumption = dlog_energy_consumption.to_frame(name='d_log_consumption')

# 5. Creating dummies for 'Easter', 'Corpus Christi', 'Subprime', 'Water_Crises' and 'Pandemic'
# 5.1 Creating 'zeros' columns
dummies = ['dummy_subprime', 'dummy_water_crisis', 'dummy_pandemic', 'dummy_easter', 'dummy_corpus_christi']

for col in dummies:
    dlog_energy_consumption[col] = 0

# 5.2 Defining Structural Breaks
# Subprime: This event had a great impact on industrial sector 
dlog_energy_consumption.loc['2008-09-01': '2009-06-30', 'dummy_subprime'] = 1

# Water Crisis
dlog_energy_consumption.loc['2014-02-01': '2015-06-30', 'dummy_water_crisis'] = 1

# Pandemic
dlog_energy_consumption.loc['2020-03-01': '2021-03-31', 'dummy_pandemic'] = 1

# 5.3 Defining 'Moving' Schedule
for Period in dlog_energy_consumption.index:
    year = Period.year

    # Dates
    e_Period = easter(year) # Easter
    cc_Period = e_Period + pd.Timedelta(days = 60) # Corpus Christi

    # If the month as the same as the holyday
    if Period.month == e_Period.month:
        dlog_energy_consumption.at[Period, 'dummy_easter'] = 1
    
    if Period.month == cc_Period.month:
        dlog_energy_consumption.at[Period, 'dummy_corpus_christi'] = 1

# 5.4 Creating 'exogenous' objects, the X, to SARIMA(X)
exog = dlog_energy_consumption[dummies]

print(exog.sum()) # Checking how many days each dummy has filled out

# 5.5 Checking if the column name is 'Consume'. If yes, use:
y = dlog_energy_consumption.iloc[:, 0] # It takes the first column.

# 6 Running the 'conservative' SARIMAX model
dlog_model = sm.tsa.statespace.SARIMAX(y, exog = exog, order = (1, 0, 1), seasonal_order = (1, 1, 1, 12), enforce_stationarity = False,
                                       enforce_invertibility = False)

dlog_result = dlog_model.fit(disp = 'off')

print(dlog_result.summary())

dlog_result.plot_diagnostics(figsize = (15, 10))
plt.savefig('dlog_energy_consumption_resids.png', dpi = 300, bbox_inches = 'tight')
plt.show()

############################################################################################################
# 7. Running the auto_arima function
# SARIMAX Model
auto_model = auto_arima(y,
                        exog = exog,
                        m = 12, # 12 months seasonality
                        seasonal = True, 
                        stepwise = True,
                        trace = True,
                        error_action = 'ignore',
                        suppress_warnings = True)

print(auto_model.summary())

# 10. VISUAL COMPARATIVE ANALYSIS
# 10.1 Rebuild the model of auto_arima function
# Order suggested: (1,0,1)x(1,0,[1,2],12) with intercept
robot_model = sm.tsa.statespace.SARIMAX(y, 
                                       order = (1, 0, 1),
                                       seasonal_order = (1, 0, 2, 12),
                                       trend = 'c')

robot_result = robot_model.fit(disp = 'off')

############################################################################################################
# 8 Manual SARIMAX Evaluation

# 8.1 Defining potential values for seasonal P, D, Q 
P = [0, 1]
D = [0, 1]
Q = [0, 1]

# Creating all possible parameter combinations
seasonal_combination = list(itertools.product(P, D, Q))

manual_results = []

for param_seasonal in seasonal_combination:
    try:
        # Adding a fix S=12 to each combination
        seasonal_order = (param_seasonal[0], param_seasonal[1], param_seasonal[2], 12)

        manual_model = sm.tsa.statespace.SARIMAX(y, order = (1, 0, 1), exog = exog, seasonal_order = seasonal_order, enforce_stationarity = False, enforce_invertibility = False)

        manual_res = manual_model.fit(disp = 'off', method = 'bfgs', maxiter = 200)

        manual_results.append({'Parameters': seasonal_order, 'AIC': manual_res.aic})
        print(f"Testing SARIMA (1, 0, 1) x {seasonal_order} -> AIC: {manual_res.aic: .3f}")

    except:
        continue

# 8.2 Converting into DataFrame to compare results
df_results = pd.DataFrame(manual_results).sort_values(by = 'AIC')
print("\n--- RANKING OF MANUAL MODELS ---")
print(df_results)

###########################################################################################################

# 9. KEEPING THE BEST MODEL 
the_best_model = sm.tsa.statespace.SARIMAX(y, exog = exog, order = (1, 0, 1), seasonal_order = (0, 1, 1, 12), enforce_stationarity = False,
                                           enforce_invertibility = False)

# Train (fit) the final model
best_result = the_best_model.fit(disp = 'off')

# Summary final check
print("Summary of the best model")
print(best_result.summary())

# Save the resids for future analysis, if necessary
best_resids = best_result.resid

###################################################################################################

# 10.2 Creating a chart
fig, axes = plt.subplots(2, 2, figsize = (18, 12))
plt.subplots_adjust(hspace = .4, wspace = .3)

# [0,0] - Main chart
axes[0, 0].plot(y, color = '#555555', label = 'Dlog Consumption (Real)')
axes[0, 0].set_title("Differentiated Times Series (Target)", fontsize = 12, fontweight = 'bold')
axes[0, 0].grid(True, alpha = .3)
axes[0,0].legend(loc = 'best')

# [0, 1] - Conservative Model
axes[0, 1].plot(dlog_result.resid, color = 'royalblue', lw = 1)
axes[0, 1].axhline(0, color = 'red', linestyle = '--', alpha = .7)
axes[0, 1].set_title(f"Resids: Conservative Model\nAIC: {dlog_result.aic: .2f}", fontsize = 11)

# [1, 0] - AUTO ARIMA - Robot Model
axes[1, 0].plot(auto_model.resid(), color = 'seagreen', lw = 1)
axes[1, 0].axhline(0, color = 'red', linestyle = '--', alpha = .7)
axes[1, 0].set_title(f"Resids: Auto ARIMA Benchmarking\nAIC: {auto_model.aic(): .2f}", fontsize = 11)

# [1, 1] - Manual Model (The best performance)
axes[1, 1].plot(best_result.resid, color = 'darkorchid', lw =1)
axes[1, 1].axhline(0, color = 'red', linestyle = '--', alpha = .7)
axes[1, 1].set_title(f"Resids: Best Model (Manual)\nAIC: {best_result.aic: .2f}", fontsize = 11)

# Title
plt.suptitle("Comparative Residual Analysis and Performance - Energy Consumption", fontsize = 16, fontweight = 'bold', y = .95)
plt.savefig('benchmarking_sarimax_resids.png', dpi = 300, bbox_inches = 'tight')
plt.show()

######################################################################

# 11. Forecasting process

# Predict scenario: next 12 months
predict_dates = pd.date_range(start = '2026-03-01', periods = 12, freq = 'MS')

# Converting into Dataframe empty exogenous variables to the future
predict_exogenous = pd.DataFrame(0, index = predict_dates, columns = dummies)

# Holidays in 2026
# Easter in April
predict_exogenous.loc['2026-04-01', 'dummy_easter'] = 1

# Corpus Christi in June
predict_exogenous.loc['2026-06-01', 'dummy_corpus_christi'] = 1

print('Predict Exogenous Table:')
print(predict_exogenous)

# 12 Prediction
forecast_object = best_result.get_forecast(steps = 12, exog = predict_exogenous)

# Evaluate the mean of prediction and the confidence interval(95%)
forecast_mean = forecast_object.predicted_mean
confidence_interval = forecast_object.conf_int()

print("\nPrediction of Consumption (Variation % dlog) for 2026:")
print(forecast_mean.tail(12))

# 12. FINAL CHART: ORIGINAL ENERGY CONSUMPTION + PREDICTION

plt.figure(figsize = (15, 8))

# 12.1 Plot the recent period
recent_period =y.loc['2022-01-01':]
plt.plot(recent_period, label = 'Original Energy Consumption (in %)', color = 'black', linewidth = 1.5)

# 12.2 Plot the prediction
plt.plot(forecast_mean, label = 'Prediction for the next 12 months (Best Result)', color = 'blue', linestyle = '--', linewidth = 1.5)

# 12.3 Add Confidence Interval
plt.fill_between(confidence_interval.index,
                 confidence_interval.iloc[:, 0],
                 confidence_interval.iloc[:, 1],
                 color = 'blue', alpha = .1, label = 'Confidence Interval (95%)')

# 12.4 Reference Line
plt.axvline(x = y.index[-1], color = 'red', linestyle = ':', label = 'Predict Start Point')
plt.axhline(y = 0, color = 'gray', linestyle = '-', alpha = .3)

plt.title('Energy Consumption Prediction: What to expect for the next 12 months?', fontsize = 16)
plt.xlabel('Year')
plt.ylabel('Monthly Variation (in %)')
plt.legend(loc = 'best')
plt.grid(True, alpha = .2)
plt.savefig('final_energy_consumption_prediction.png', dpi = 300, bbox_inches = 'tight')
plt.show()

# 13. CONVERTING VARIATION INTO GWK
# 13.1 Taking the last value from original serie in log 
last_real_log = log_energy_consumption.iloc[-1]

# 13.2 Rescaling and calculating cumulative sums
predict_log = (forecast_mean/100).cumsum()

# 13.3 Add to the last value to take the same standard of the log
final_predict_log = last_real_log + predict_log

# 13.4 Apply the exponential function
gwh_prediction = np.exp(final_predict_log)

print("--- REAL CONSUMPTION PREDICTION IN GWh (2026) ---")
print(gwh_prediction.tail(12))

# 14 ANNUAL COMPARATIVE: 2025 (REAL) VS. NEXT 12 MONTHS (FORECASTED)

# 14.1 Calculating the 2025 monthly mean (original series)
mean_2025 = energy_consumption.loc['2025-03-01': '2026-02-28'].mean().values[0]

# 14.2 Calculating the 12 monthly mean (predicted data)
mean_next_12_months = gwh_prediction.mean()

# 14.3 Calculating the annual growth (in %)
growth = ((mean_next_12_months/mean_2025)-1)*100

# 14.4 Identifying predicted peaks and troughs
peak_month = gwh_prediction.idxmax().strftime('%B/%Y')
peak_value = gwh_prediction.max()
trough_month = gwh_prediction.idxmin().strftime('%B/%Y')
trough_value = gwh_prediction.min()

print("FINAL RESULT")
print(f"2025 Monthly Mean: {mean_2025/1e6:.2f} million GWh")
print(f"Next 12-Month Average Forecast: {mean_next_12_months/1e6:.2f} million GWh")
print(f"Projected 12-Month Growth: {growth: .2f}%")
print(f"Predicted Peak Demand: {peak_value/1e6:.2f} million GWh in {peak_month}")
print(f"Predicted Trough Demand: {trough_value/1e6:.2f} million GWh in {trough_month}")
